In [1]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

df = spark.read.format("mongodb").load()



In [2]:
# Ver estructura de columnas
df.printSchema()

# Ver las primeras 5 filas
df.show(5)


root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+--------------------+-------------------+------+--------------------+-----+
|                 _id|           categoria|      fecha_captura| grupo|                item|valor|
+--------------------+--------------------+-------------------+------+--------------------+-----+
|69e947cab0c520883...|Energía y Sustent...|2026-04-22 22:12:26|Energy|  It’s Only Himalaya|45.17|
|69e947cab0c520883...|Energía y Sustent...|2026-04-22 22:12:26|Energy|Full Moon over No...|49.43|
+--------------------+--------------------+-------------------+------+--------------------+-----+



In [4]:
from pyspark.sql.functions import col

# Filtrar registros válidos
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros válidos:", df_filtrado.count())


PASO 2: Limpieza completada.
Registros originales: 2
Registros válidos: 2


In [5]:
df.select("item", "valor").show()


+--------------------+-----+
|                item|valor|
+--------------------+-----+
|  It’s Only Himalaya|45.17|
|Full Moon over No...|49.43|
+--------------------+-----+



In [6]:
df.filter(df["valor"] > 100).show()


+---+---------+-------------+-----+----+-----+
|_id|categoria|fecha_captura|grupo|item|valor|
+---+---------+-------------+-----+----+-----+
+---+---------+-------------+-----+----+-----+



In [7]:
df.groupBy("grupo").count().show()


+------+-----+
| grupo|count|
+------+-----+
|Energy|    2|
+------+-----+



In [8]:
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANÁLISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()


ANÁLISIS DE MERCADO POR CATEGORIA:
+--------------------+---------------+---------------+-------------+-------------+
|           categoria|cantidad_libros|precio_promedio|precio_minimo|precio_maximo|
+--------------------+---------------+---------------+-------------+-------------+
|Energía y Sustent...|              2|           47.3|        45.17|        49.43|
+--------------------+---------------+---------------+-------------+-------------+



In [12]:
from pyspark.sql import functions as F

ganador = df.orderBy(F.desc("valor")).limit(1)

ganador.select("item", "valor", "categoria", "fecha_captura").show()

+--------------------+-----+--------------------+-------------------+
|                item|valor|           categoria|      fecha_captura|
+--------------------+-----+--------------------+-------------------+
|Full Moon over No...|49.43|Energía y Sustent...|2026-04-22 22:12:26|
+--------------------+-----+--------------------+-------------------+



In [19]:
from pyspark.sql import functions as F

NOMBRE_GRUPO = "Energy"

ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()


--- TICKET DE SALIDA: Energy ---
+--------------------+------------+------------+---------------------+
|           categoria|total_libros|precio_medio|ultima_sincronizacion|
+--------------------+------------+------------+---------------------+
|Energía y Sustent...|           2|        47.3|  2026-04-22 22:12:26|
+--------------------+------------+------------+---------------------+

